In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!cp -r /content/drive/MyDrive/split_dataset_balanced /content/

In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
Tesla T4


In [6]:
train_dir = '/content/split_dataset_balanced/train'
val_dir = '/content/split_dataset_balanced/val'

print(os.path.exists(train_dir), train_dir)
print(os.path.exists(val_dir), val_dir)

True /content/split_dataset_balanced/train
True /content/split_dataset_balanced/val


In [7]:
img_size = 128
batch_size = 32

train_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

In [8]:
train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds = datasets.ImageFolder(val_dir, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

class_names = train_ds.classes
print(class_names)
print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))

['destroyed', 'major-damage', 'minor-damage', 'no-damage']
Train samples: 2064
Val samples: 520


In [9]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [10]:
model = SimpleCNN(num_classes=4).to(device)
print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=4, bias=True)
  )
)


In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

In [12]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')

    return epoch_loss, epoch_acc, epoch_f1


def validate_one_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')

    return epoch_loss, epoch_acc, epoch_f1, all_labels, all_preds

In [13]:
num_epochs = 5
best_f1 = 0.0
save_path = '/content/best_cnn_balanced.pth'

for epoch in range(num_epochs):
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_f1, y_true, y_pred = validate_one_epoch(model, val_loader, criterion)

    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), save_path)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val   F1: {val_f1:.4f}")
    print("-" * 60)

Epoch 1/5
Train Loss: 1.3038 | Train Acc: 0.3702 | Train F1: 0.3438
Val   Loss: 1.1994 | Val   Acc: 0.4346 | Val   F1: 0.3234
------------------------------------------------------------
Epoch 2/5
Train Loss: 1.1992 | Train Acc: 0.4511 | Train F1: 0.4278
Val   Loss: 1.1287 | Val   Acc: 0.5019 | Val   F1: 0.4250
------------------------------------------------------------
Epoch 3/5
Train Loss: 1.1552 | Train Acc: 0.4806 | Train F1: 0.4728
Val   Loss: 1.0894 | Val   Acc: 0.5308 | Val   F1: 0.5015
------------------------------------------------------------
Epoch 4/5
Train Loss: 1.1323 | Train Acc: 0.4961 | Train F1: 0.4907
Val   Loss: 1.1122 | Val   Acc: 0.5308 | Val   F1: 0.5113
------------------------------------------------------------
Epoch 5/5
Train Loss: 1.0863 | Train Acc: 0.5252 | Train F1: 0.5232
Val   Loss: 1.0586 | Val   Acc: 0.5404 | Val   F1: 0.5126
------------------------------------------------------------


In [14]:
model.load_state_dict(torch.load(save_path))

_, val_acc, val_f1, y_true, y_pred = validate_one_epoch(model, val_loader, criterion)

precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print("Best Validation Accuracy:", val_acc)
print("Best Validation Precision:", precision)
print("Best Validation Recall:", recall)
print("Best Validation Macro F1:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Best Validation Accuracy: 0.5403846153846154
Best Validation Precision: 0.6184831825100674
Best Validation Recall: 0.5403846153846155
Best Validation Macro F1: 0.5126407888912907

Classification Report:

              precision    recall  f1-score   support

   destroyed       0.86      0.69      0.77       130
major-damage       0.39      0.89      0.55       130
minor-damage       0.59      0.13      0.21       130
   no-damage       0.64      0.45      0.52       130

    accuracy                           0.54       520
   macro avg       0.62      0.54      0.51       520
weighted avg       0.62      0.54      0.51       520


Confusion Matrix:

[[ 90  24   3  13]
 [  6 116   1   7]
 [  5  95  17  13]
 [  4  60   8  58]]


In [15]:
model.load_state_dict(torch.load(save_path))

_, val_acc, val_f1, y_true, y_pred = validate_one_epoch(model, val_loader, criterion)

precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print("Best Validation Accuracy:", val_acc)
print("Best Validation Precision:", precision)
print("Best Validation Recall:", recall)
print("Best Validation Macro F1:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Best Validation Accuracy: 0.5403846153846154
Best Validation Precision: 0.6184831825100674
Best Validation Recall: 0.5403846153846155
Best Validation Macro F1: 0.5126407888912907

Classification Report:

              precision    recall  f1-score   support

   destroyed       0.86      0.69      0.77       130
major-damage       0.39      0.89      0.55       130
minor-damage       0.59      0.13      0.21       130
   no-damage       0.64      0.45      0.52       130

    accuracy                           0.54       520
   macro avg       0.62      0.54      0.51       520
weighted avg       0.62      0.54      0.51       520


Confusion Matrix:

[[ 90  24   3  13]
 [  6 116   1   7]
 [  5  95  17  13]
 [  4  60   8  58]]


In [16]:
print("\nFINAL METRICS")
print(f"Accuracy  : {val_acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")


FINAL METRICS
Accuracy  : 0.5404
Precision : 0.6185
Recall    : 0.5404
F1 Score  : 0.5126
